# 🚀 Roblox Asset AI - Specialized Model Training & Serving
### Dedicated 3D Model & Animation LLM (Runs on Kaggle Free T4 GPU / 16GB VRAM)

This notebook builds a **compact, highly specialized AI model** that **ONLY knows Roblox Studio**:
- **NO general chat, NO coding outside Roblox, NO fluff**
- Generates production-ready **Roblox 3D Model Intermediate Representation (IR)**
- Generates valid **Roblox Animation IR (Keyframes & Joint Transforms)**
- Lightweight base model (`Qwen2.5-Coder-1.5B` or `Llama-3.2-1B`) fine-tuned with **4-bit QLoRA**
- Runs in < 15 minutes on a free T4 GPU
- Hosts a live REST API so your Roblox Asset AI web application can connect to Kaggle directly!

In [ ]:
# Cell 1: Verify Hardware & GPU Acceleration
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# Cell 2: Install Essential Dependencies
!pip install -q transformers peft accelerate bitsandbytes datasets fastapi uvicorn pyngrok

In [ ]:
# Cell 3: Generate or Verify the Specialized Roblox Dataset
!python scripts/generate_synthetic_dataset.py

import json
with open("./dataset/roblox_sft_train.jsonl", "r") as f:
    first = json.loads(f.readline())
    print("Sample Task:", first["task"])
    print("Sample Prompt:", first["prompt"])
    print("Sample Output Preview:", first["messages"][2]["content"][:200] + "...")

In [ ]:
# Cell 4: Launch Model Fine-Tuning with 4-bit QLoRA
!python kaggle/train.py \
    --model_name "Qwen/Qwen2.5-Coder-1.5B-Instruct" \
    --data_dir "./dataset" \
    --output_dir "./checkpoints/roblox-asset-ai-t4" \
    --batch_size 2 \
    --grad_accum 4 \
    --epochs 3 \
    --lr 2e-4 \
    --use_4bit

In [ ]:
# Cell 5: Inspect Evaluation Results & Save Compressed Weights
import json
eval_path = "./checkpoints/roblox-asset-ai-t4/eval_results.json"
try:
    with open(eval_path, "r") as f:
        print("Training Metrics:", json.dumps(json.load(f), indent=2))
except Exception as e:
    print(e)

!tar -czvf roblox_asset_ai_lora.tar.gz ./checkpoints/roblox-asset-ai-t4

In [ ]:
# Cell 6: Launch Dedicated Inference API Server
# Set your free ngrok auth token here to get a public HTTPS URL (from ngrok.com)
NGROK_TOKEN = ""  # Optional: paste your ngrok authtoken here

import subprocess
cmd = ["python", "kaggle/serve.py", "--port", "8000"]
if NGROK_TOKEN:
    cmd.extend(["--ngrok_token", NGROK_TOKEN])

print("Starting Roblox Asset AI Kaggle Server...")
subprocess.run(cmd)